# Comparación de fine-tuning por estrategia de prompting

Compara Qwen base y los adapters LoRA Zero-shot, One-shot y Few-shot. Reproduce el split `test_size=0.2, random_state=42`, usa `ContextRetriever` exclusivamente local y evalúa cada adapter con el prompt guardado durante su entrenamiento.

**Importante:** ejecuta desde la raíz del repositorio. No se usa `FallbackRetriever` ni búsqueda web.

## 1. Imports y configuración

In [9]:
import gc, json, sys
from pathlib import Path
import pandas as pd
import torch
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    found = next((p for p in PROJECT_ROOT.parents if (p / "src").exists()), None)
    if found is None:
        raise FileNotFoundError("No encuentro la raíz del repositorio; ajusta PROJECT_ROOT.")
    PROJECT_ROOT = found
sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluate import evaluate_predictions
from src.preprocess.preprocess import prepare_dataset
from src.retrieval.context_retriever import ContextRetriever

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
SEED = 42
MAX_LENGTH = 384
MAX_NEW_TOKENS = 32
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

INDEX_PATH = PROJECT_ROOT / "models" / "context_embeddings.pt"
ADAPTERS_DIR = PROJECT_ROOT / "models" / "prompt_training"
PROMPT_RESULTS_DIR = PROJECT_ROOT / "results" / "prompt_training"
OUTPUT_DIR = PROJECT_ROOT / "results" / "prompt_training_comparison"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ADAPTER_PATHS = {
    "FT-Zero": ADAPTERS_DIR / "qwen_lora_zero",
    "FT-One": ADAPTERS_DIR / "qwen_lora_one",
    "FT-Few": ADAPTERS_DIR / "qwen_lora_few",
}
PROMPT_KEYS = {"FT-Zero": "zero", "FT-One": "one", "FT-Few": "few"}

print("Proyecto:", PROJECT_ROOT)
print("Dispositivo:", DEVICE)
print("Índice local:", INDEX_PATH.exists())
for name, path in ADAPTER_PATHS.items():
    print(f"{name}: {path} | existe={path.exists()}")

Proyecto: C:\Users\chech\Practica_PLN_II
Dispositivo: cuda
Índice local: True
FT-Zero: C:\Users\chech\Practica_PLN_II\models\prompt_training\qwen_lora_zero | existe=True
FT-One: C:\Users\chech\Practica_PLN_II\models\prompt_training\qwen_lora_one | existe=True
FT-Few: C:\Users\chech\Practica_PLN_II\models\prompt_training\qwen_lora_few | existe=True


## 2. Cargar dataset y reproducir el split

In [10]:
dataset = prepare_dataset(
    str(PROJECT_ROOT / "data" / "questions_train.json"),
    str(PROJECT_ROOT / "data" / "contexts.json"),
)
train_data, test_data = train_test_split(
    dataset, test_size=0.2, random_state=SEED
)
train_data, test_data = list(train_data), list(test_data)
print(f"Train: {len(train_data)} | Test: {len(test_data)}")

Train: 280 | Test: 70


## 3. Recuperación local de contextos

In [11]:
retriever = ContextRetriever(str(INDEX_PATH), device=DEVICE)

def resolve_contexts(samples):
    resolved = []
    for i, item in enumerate(samples, 1):
        result = retriever.retrieve(item["question"])
        resolved.append({
            **item,
            "context": result["context"],
            "retrieved_context_id": result["context_id"],
            "retrieval_score": float(result["score"]),
        })
        if i % 25 == 0 or i == len(samples):
            print(f"Contextos locales: {i}/{len(samples)}")
    return resolved

resolved_test_data = resolve_contexts(test_data)
print("Test preparado:", len(resolved_test_data))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Contextos locales: 25/70
Contextos locales: 50/70
Contextos locales: 70/70
Test preparado: 70


## 4. Cargar los prompts exactos usados en entrenamiento

In [12]:
PROMPTS = {}
for model_label, prompt_key in PROMPT_KEYS.items():
    path = PROMPT_RESULTS_DIR / prompt_key / "training_prompt.txt"
    if not path.exists():
        raise FileNotFoundError(
            f"No existe {path}. Comprueba que el entrenamiento guardó "
            "training_prompt.txt en results/prompt_training/<zero|one|few>/."
        )
    PROMPTS[model_label] = path.read_text(encoding="utf-8").strip()
    print(f"{model_label}: {len(PROMPTS[model_label])} caracteres")

for name, prompt in PROMPTS.items():
    print(f"\n--- {name} ---\n{prompt[:350]}{'...' if len(prompt) > 350 else ''}")

FT-Zero: 147 caracteres
FT-One: 1152 caracteres
FT-Few: 4087 caracteres

--- FT-Zero ---
Answer the question using the context when available. If the answer is unknown or the question is unanswerable, return an empty answer. Be concise.

--- FT-One ---
You are a precise extractive question-answering system.

Use the provided context as the only source of information. Answer the question
with the shortest complete answer explicitly supported by the context.

Do not use outside knowledge, guesses, or information from similar entities.

Return only the answer, with no explanation, introduction, labe...

--- FT-Few ---
You are a precise extractive question-answering system.

Use the provided context as the only source of information. Answer the question
with the shortest complete answer explicitly supported by the context.

Do not use outside knowledge, guesses, or information from similar entities.

Return only the answer, with no explanation, introduction, labe...


## 5. Tokenizer, generación y carga de modelos

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def make_messages(item, system_prompt):
    context = item.get("context") or "No context is available."
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {item['question']}"},
    ]

def generate_answer(model, item, system_prompt):
    prompt = tokenizer.apply_chat_template(
        make_messages(item, system_prompt),
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=MAX_LENGTH
    )
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated = outputs[:, inputs["input_ids"].shape[1]:]
    return tokenizer.batch_decode(generated, skip_special_tokens=True)[0].strip()

def load_base_model():
    def load_base_model():
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    dtype = (
        torch.bfloat16
        if DEVICE.type == "cuda"
        else torch.float32
    )

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=dtype,
        device_map="auto",
        max_memory={
            0: "12GiB",
            "cpu": "32GiB"
        }
    )

    model.eval()
    return model

def load_adapter(path):
    if not path.exists():
        raise FileNotFoundError(
            f"No encuentro el adapter: {path}"
        )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    dtype = (
        torch.bfloat16
        if DEVICE.type == "cuda"
        else torch.float32
    )

    base = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=dtype,
    )

    model = PeftModel.from_pretrained(
        base,
        str(path),
        is_trainable=False,
    )

    model.eval()
    return model

def unload_model(model):
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 6. Función de evaluación

In [14]:
def evaluate_condition(model, label, prompt_name, system_prompt, samples):
    print(f"\n[{label}] prompt={prompt_name}")
    predictions = []
    for i, item in enumerate(samples, 1):
        pred = generate_answer(model, item, system_prompt)
        predictions.append({
            "question": item["question"],
            "prediction": pred,
            "reference": item.get("answer"),
            "has_answer": bool(item.get("has_answer")),
            "retrieved_context_id": item.get("retrieved_context_id"),
            "retrieval_score": item.get("retrieval_score"),
        })
        if i % 10 == 0 or i == len(samples):
            print(f"  {i}/{len(samples)}")

    answerable = [x for x in predictions if x["has_answer"]]
    unanswerable = [x for x in predictions if not x["has_answer"]]
    metrics = {
        "overall": evaluate_predictions(predictions),
        "answerable": evaluate_predictions(answerable),
        "unanswerable": evaluate_predictions(unanswerable),
    }
    print(
        f"EM={metrics['overall']['exact_match']}% | "
        f"F1={metrics['overall']['f1_score']}% | "
        f"Answerable F1={metrics['answerable']['f1_score']}% | "
        f"Unanswerable F1={metrics['unanswerable']['f1_score']}%"
    )
    return {"model": label, "prompt": prompt_name, "metrics": metrics, "predictions": predictions}

all_results = []

## 7. Evaluar Qwen base con cada prompt

In [15]:
base_model = load_base_model()
for label, prompt in [
    ("Base-Zero", PROMPTS["FT-Zero"]),
    ("Base-One", PROMPTS["FT-One"]),
    ("Base-Few", PROMPTS["FT-Few"]),
]:
    all_results.append(evaluate_condition(
        base_model, label, label.replace("Base-", "").lower(),
        prompt, resolved_test_data
    ))
unload_model(base_model)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 1.02 GiB. GPU 0 has a total capacity of 15.93 GiB of which 0 bytes is free. Of the allocated memory 29.64 GiB is allocated by PyTorch, and 232.00 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

## 8. Evaluar cada adapter con su prompt correspondiente

In [ ]:

for model_label, prompt_key in PROMPT_KEYS.items():
    print(
        "\n" + "=" * 65
        + f"\nEvaluando {model_label}\n"
        + "=" * 65
    )

    model = load_adapter(
        ADAPTER_PATHS[model_label]
    )

    try:
        result = evaluate_condition(
            model,
            model_label,
            prompt_key,
            PROMPTS[model_label],
            resolved_test_data,
        )

        all_results.append(result)

    finally:
        del model
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()



Evaluando FT-Zero


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]


[FT-Zero] prompt=zero
  10/70
  20/70
  30/70
  40/70
  50/70
  60/70
  70/70
EM=75.71% | F1=79.64% | Answerable F1=75.46% | Unanswerable F1=93.75%

Evaluando FT-One


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 130.00 MiB. GPU 0 has a total capacity of 15.93 GiB of which 0 bytes is free. Of the allocated memory 29.64 GiB is allocated by PyTorch, and 239.00 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

## 9. Resumen de métricas

In [ ]:
rows = []
for result in all_results:
    m = result["metrics"]
    rows.append({
        "Modelo": result["model"],
        "Prompt": result["prompt"],
        "EM (%)": m["overall"]["exact_match"],
        "F1 (%)": m["overall"]["f1_score"],
        "Answerable F1 (%)": m["answerable"]["f1_score"],
        "Unanswerable F1 (%)": m["unanswerable"]["f1_score"],
    })
summary_df = pd.DataFrame(rows)
display(summary_df.round(2))
summary_path = OUTPUT_DIR / "summary.csv"
summary_df.to_csv(summary_path, index=False)
print("Resumen guardado en:", summary_path)

## 10. Gráfico comparativo

In [ ]:
ax = summary_df.set_index("Modelo")[[
    "F1 (%)", "Answerable F1 (%)", "Unanswerable F1 (%)"
]].plot(kind="bar", figsize=(12, 6))
ax.set_title("Qwen base vs adapters LoRA por estrategia de prompting")
ax.set_ylabel("F1 (%)")
ax.set_xlabel("")
ax.set_ylim(0, 100)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 11. Guardar predicciones completas

In [ ]:
results_path = OUTPUT_DIR / "predictions_and_metrics.json"
with open(results_path, "w", encoding="utf-8") as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)
print("Resultados completos:", results_path)
print("Resumen:", summary_path)